# Monitoraggio della reputazione online di un'azienda

> **Azienda**: MachineInnovators Inc. — leader nello sviluppo di applicazioni di machine learning scalabili e pronte per la produzione
> **Problema**: monitorare manualmente il sentiment degli utenti sui social media e' inefficiente, soggetto a errori umani e troppo lento per intervenire in tempo su un calo di reputazione
> **Soluzione proposta**: automatizzare l'analisi del sentiment con un modello pre-addestrato e costruire attorno ad esso una base MLOps di monitoraggio continuo, alerting e retraining
> **Modello richiesto**: [`cardiffnlp/twitter-roberta-base-sentiment-latest`](https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment-latest)
> **Dataset pubblico**: [`tweet_eval`, task `sentiment`](https://huggingface.co/datasets/cardiffnlp/tweet_eval)

## Contesto

Il modo in cui un'azienda viene percepita sui social media influenza direttamente vendite, fiducia degli investitori e valore del brand. Il volume di menzioni (tweet, post, recensioni) che un'azienda di medie o grandi dimensioni riceve ogni giorno rende impossibile una lettura manuale sistematica: servono strumenti automatici capaci di leggere grandi quantita' di testo e restituire un giudizio sintetico e strutturato — questo e' il ruolo dell'NLP in questo progetto.

MachineInnovators Inc. vuole integrare metodologie **MLOps** per non fermarsi alla singola previsione, ma costruire un flusso completo che copra lo sviluppo, l'implementazione, il monitoraggio continuo e il retraining del modello di analisi del sentiment. L'obiettivo di business e' abilitare l'azienda a migliorare e monitorare la propria reputazione sui social media in modo tempestivo — rilevando un peggioramento del sentiment mentre e' ancora gestibile, non dopo che e' gia' diventato un problema pubblico.

## Obiettivo del notebook

1. Caricare `tweet_eval/sentiment`, un dataset pubblico di tweet etichettati come `negative`/`neutral`/`positive`, e controllarne la qualita' (valori mancanti, duplicati, bilanciamento delle classi).
2. Caricare il modello pre-addestrato richiesto (`cardiffnlp/twitter-roberta-base-sentiment-latest`) ed eseguire l'inferenza su un campione del test set, **senza fine-tuning**.
3. Valutare le prestazioni con metriche adatte a un problema multiclasse potenzialmente sbilanciato: accuracy, precision/recall/F1 macro e weighted, matrice di confusione, confidence.
4. Instradare a revisione umana i casi a bassa confidence, invece di considerare ogni previsione ugualmente affidabile.
5. Simulare un sistema di **monitoraggio continuo** del sentiment nel tempo, con alert basati su una baseline storica e su una soglia di incremento configurabile.
6. Definire regole di **monitoraggio del modello e retraining**, distinguendo esplicitamente un calo di qualita' del modello da una vera crisi reputazionale.
7. Proporre una bozza di **pipeline CI/CD** (test automatici + GitHub Actions) per portare il sistema verso un contesto di produzione, con il link al repository GitHub pubblico richiesto dalla consegna.

## Metodologia

Il modello richiesto e' un Transformer (RoBERTa) gia' specializzato per la sentiment analysis su Twitter: viene usato **in inferenza diretta, senza fine-tuning**, e valutato sul benchmark pubblico `tweet_eval`. Il valore aggiunto del notebook non sta quindi nell'addestrare un classificatore da zero, ma nel valutare criticamente le prestazioni del modello gia' pronto e nel progettare il "contorno" operativo — data quality, metriche, monitoraggio, alerting, retraining, CI/CD — che serve per usarlo in modo responsabile in un contesto aziendale.

Tutti i parametri di esecuzione e le soglie operative (batch size, numero di esempi valutati, F1 minimo accettabile, confidence minima, soglia di revisione umana, incremento massimo di sentiment negativo) sono centralizzati in un'unica `Config`: un solo punto da modificare invece di costanti sparse nel notebook. La stessa logica di alert (`evaluate_negative_share_alert`) viene riutilizzata sia sui dati reali sia su uno scenario dimostrativo sintetico, per verificare che scatti correttamente quando serve davvero, e non solo sulla carta.

## Struttura del notebook

0. Link al repository GitHub del progetto
1. Installazione librerie
2. Importazioni e configurazione centralizzata (`Config`)
3. Caricamento del dataset (`tweet_eval/sentiment`)
4. Controllo qualita' dei dati
5. Analisi esplorativa (distribuzione delle classi, lunghezza dei testi, esempi)
6. Caricamento del modello pre-addestrato
7. Funzioni modulari di inference, valutazione e alert
8. Inference sul test set
9. Valutazione delle performance (metriche, matrice di confusione, confidence)
10. Instradamento a revisione umana (human-in-the-loop)
11. Monitoraggio continuo della reputazione (simulazione temporale + alert, con dimostrazione su scenario sintetico)
12. Monitoraggio modello e retraining (regole e soglie)
13. Pipeline CI/CD proposta
14. Demo facoltativa con Gradio
15. Conclusioni finali

Questo notebook e' pensato per essere eseguito su **Google Colab**.

## 0. Link repository GitHub

La consegna richiede che il notebook contenga il link al repository GitHub pubblico. Dopo aver creato il repository e caricato i file, sostituire il placeholder qui sotto.

In [ ]:
# Link al repository GitHub del progetto.
# Sostituire questa stringa con il link reale prima della consegna.
GITHUB_REPOSITORY_URL = "INSERIRE_QUI_IL_LINK_AL_REPOSITORY_GITHUB"

print(f"Repository GitHub progetto: {GITHUB_REPOSITORY_URL}")

## 1. Installazione librerie

In [ ]:
!pip install -q transformers datasets evaluate accelerate scikit-learn matplotlib seaborn pandas numpy gradio huggingface_hub

## 2. Importazioni e configurazione

In [ ]:
# ============================================================
# IMPORTAZIONI E CONFIGURAZIONE
# ============================================================

import os
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
)

sns.set_theme(style="whitegrid")

In [ ]:
# ============================================================
# CELLA — Config: parametri centralizzati del progetto
# ============================================================
@dataclass
class Config:
    seed: int = 42
    model_name: str = "cardiffnlp/twitter-roberta-base-sentiment-latest"
    dataset_name: str = "cardiffnlp/tweet_eval"
    dataset_task: str = "sentiment"
    max_eval_samples: int = 1000
    batch_size: int = 32
    label_map: Dict[int, str] = None

    # Soglie operative di monitoraggio/retraining/revisione umana.
    min_macro_f1: float = 0.70
    min_average_confidence: float = 0.60
    max_negative_share_increase: float = 0.15
    low_confidence_review_threshold: float = 0.60
    n_current_weeks_for_monitoring: int = 1

    def __post_init__(self):
        if self.label_map is None:
            # tweet_eval/sentiment usa questa codifica ufficiale:
            # 0 = negative, 1 = neutral, 2 = positive.
            self.label_map = {0: "negative", 1: "neutral", 2: "positive"}

cfg = Config()

In [ ]:
# ============================================================
# Impostazione del Seed
# ============================================================
def set_seed(seed: int = 42) -> None:
    """Rende piu' riproducibili campionamento e risultati."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(cfg.seed)
device = 0 if torch.cuda.is_available() else -1
print("Device usato:", "GPU" if device == 0 else "CPU")

## 3. Caricamento del dataset

Il dataset scelto e' `cardiffnlp/tweet_eval` (subtask `sentiment`), una raccolta pubblica di tweet etichettati come negativi, neutri o positivi. E' coerente con il modello CardiffNLP richiesto, perche' entrambi lavorano sul linguaggio tipico di Twitter/social media — non a caso condividono anche lo stesso namespace su HuggingFace Hub.

In [ ]:
# ============================================================
# CARICAMENTO DATASET PUBBLICO
# ============================================================

raw_dataset = load_dataset(cfg.dataset_name, cfg.dataset_task)

print(raw_dataset)
print("\nSplit disponibili:", list(raw_dataset.keys()))

In [ ]:
train_df = raw_dataset["train"].to_pandas()
val_df = raw_dataset["validation"].to_pandas()
test_df = raw_dataset["test"].to_pandas()

for df in [train_df, val_df, test_df]:
    df["sentiment"] = df["label"].map(cfg.label_map)
    df["text_length"] = df["text"].str.len()
    df["word_count"] = df["text"].str.split().str.len()

print("Dimensioni train/validation/test:")
print(train_df.shape, val_df.shape, test_df.shape)

train_df.head()

## 4. Controllo qualita' dei dati

Prima di usare un modello e' utile controllare valori mancanti, duplicati e distribuzione delle etichette. 

In [ ]:
# ============================================================
# DATA QUALITY CHECK
# ============================================================

def data_quality_report(df: pd.DataFrame, name: str) -> None:
    print(f"--- {name.upper()} ---")
    print(f"Righe: {len(df):,}")
    print("Valori mancanti:")
    print(df.isnull().sum())
    print(f"Duplicati sul testo: {df['text'].duplicated().sum():,}")
    print("Distribuzione label:")
    print(df["sentiment"].value_counts().to_string())
    print()

data_quality_report(train_df, "train")
data_quality_report(val_df, "validation")
data_quality_report(test_df, "test")

## 5. Analisi esplorativa

Questa sezione serve a capire la forma dei dati prima della valutazione del modello: quante classi ci sono, quanto sono bilanciate e quanto sono lunghi i testi. Sono informazioni semplici, ma aiutano molto a interpretare i risultati successivi.

In [ ]:
# ============================================================
# GRAFICO 1: DISTRIBUZIONE DELLE CLASSI
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (name, df) in zip(axes, [("Train", train_df), ("Validation", val_df), ("Test", test_df)]):
    order = ["negative", "neutral", "positive"]
    counts = df["sentiment"].value_counts().reindex(order)
    sns.barplot(x=counts.index, y=counts.values, ax=ax, palette="Set2")
    ax.set_title(f"Distribuzione sentiment - {name}")
    ax.set_xlabel("Sentiment")
    ax.set_ylabel("Numero testi")
    for i, val in enumerate(counts.values):
        ax.text(i, val + max(counts.values) * 0.02, f"{val:,}", ha="center", fontsize=9)

plt.tight_layout()
plt.show()

## Analisi della qualità e della distribuzione del dataset

### Controllo degli split

Il dataset è suddiviso in:

- **Train:** 45.615 esempi
- **Validation:** 2.000 esempi
- **Test:** 12.284 esempi

Il controllo preliminare non evidenzia **valori mancanti** nelle variabili analizzate (`text`, `label`, `sentiment`, `text_length`, `word_count`) in nessuno dei tre split.

Sono stati rilevati **29 testi duplicati nel Train**, mentre Validation e Test non presentano duplicati. Considerata la dimensione del training set, si tratta comunque di una quota molto ridotta.

### Distribuzione delle classi

| Split | Negative | Neutral | Positive |
|---|---:|---:|---:|
| **Train** | 7.093 (15,5%) | 20.673 (45,3%) | 17.849 (39,1%) |
| **Validation** | 312 (15,6%) | 869 (43,5%) | 819 (41,0%) |
| **Test** | 3.972 (32,3%) | 5.937 (48,3%) | 2.375 (19,3%) |

Guardando le percentuali, Train e Validation si assomigliano molto: in entrambi la classe più comune è `neutral`, poi `positive`, e `negative` è quella con meno esempi.

Il **Test set presenta invece una composizione sensibilmente diversa**. 

La classe `negative` passa da circa **15–16% a 32,3%**, quindi quasi raddoppia, mentre `positive` scende da circa **39–41% a 19,3%**, circa la metà. `Neutral` rimane invece la classe più rappresentata in tutti gli split.

→ È quindi presente un **distribution shift nelle proporzioni delle classi tra Train/Validation e Test**.

Questo aspetto dovrà essere considerato nell'interpretazione delle metriche finali, perché il modello verrà valutato su una distribuzione delle label diversa da quella osservata durante training e validation.

In [ ]:
# ============================================================
# GRAFICO 2: LUNGHEZZA DEI TESTI PER SENTIMENT
# ============================================================

plt.figure(figsize=(10, 5))
sns.boxplot(data=train_df, x="sentiment", y="word_count", order=["negative", "neutral", "positive"], palette="Set2")
plt.title("Distribuzione della lunghezza dei testi per sentiment")
plt.xlabel("Sentiment")
plt.ylabel("Numero parole")
plt.tight_layout()
plt.show()

### Osservazioni sulla lunghezza dei testi

Dal grafico si può osservare che la **lunghezza dei testi è abbastanza simile nelle tre classi di sentiment**.

- I testi `negative` hanno una lunghezza mediana leggermente maggiore, circa **21 parole**.
- I testi `neutral` e `positive` hanno invece una mediana di circa **19 parole**.
- La maggior parte dei testi, per tutte le classi, si concentra indicativamente tra **16 e 24 parole**.
- Sono presenti alcuni **valori anomali (outlier)**, soprattutto testi molto brevi con meno di 5-6 parole e alcuni testi più lunghi, intorno alle 34-35 parole.

Nel complesso, non si notano grandi differenze nella lunghezza dei testi tra `negative`, `neutral` e `positive`.

→ La **lunghezza del testo non sembra quindi essere una caratteristica che distingue in modo evidente le tre classi**: il sentiment dovrà essere riconosciuto principalmente a partire dal contenuto delle frasi e non semplicemente dal numero di parole.

In [ ]:
# ============================================================
# GRAFICO 3: ESEMPI DI TESTI PER CLASSE
# ============================================================

for sentiment in ["negative", "neutral", "positive"]:
    print(f"\n--- Esempi classe {sentiment.upper()} ---")
    sample_texts = train_df[train_df["sentiment"] == sentiment].sample(3, random_state=cfg.seed)["text"].tolist()
    for idx, text in enumerate(sample_texts, start=1):
        print(f"{idx}. {text}")

### Osservazioni sugli esempi testuali

Osservando alcuni esempi casuali per ciascuna classe si nota che i testi hanno le caratteristiche tipiche di messaggi provenienti da **Twitter/X**: sono brevi, informali e possono contenere **hashtag, menzioni (`@user`), abbreviazioni, nomi propri e punteggiatura usata per enfatizzare il messaggio**.

Negli esempi `negative` sono presenti espressioni che comunicano chiaramente insoddisfazione o frustrazione, mentre nei `positive` compaiono messaggi con un tono più favorevole o entusiasta. I testi `neutral` risultano invece più descrittivi o informativi e non esprimono un'opinione particolarmente positiva o negativa.

Si nota inoltre che il sentiment non dipende necessariamente da singole parole, ma dal **significato complessivo della frase e dal contesto**.

→ Questi esempi mostrano quindi che la classificazione del sentiment richiede di comprendere il contenuto del testo, tenendo conto anche del linguaggio informale tipico dei social network.

## 6. Caricamento del modello pre-addestrato

Il modello richiesto e' una versione RoBERTa addestrata per sentiment analysis su Twitter. Usiamo una pipeline HuggingFace per tenere il codice leggibile e adatto a Colab.

In [ ]:
# ============================================================
# MODELLO HUGGINGFACE
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(cfg.model_name)
model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name)

# Creazione una pipeline Hugging Face per la classificazione del sentiment.
# La pipeline si occupa automaticamente di:
# - tokenizzare il testo con il tokenizer scelto;
# - passare gli input al modello;
# - eseguire la previsione;
# - restituire la classe prevista con il relativo score.
#
# `task` indica a Hugging Face quale operazione vogliamo eseguire.
# Alcuni esempi:
# - task="text-classification"     -> classificazione generica di un testo
#   es. "This email is spam" -> spam
# - task="text-generation"         -> generazione di testo
#   es. "Once upon a time..." -> continua la frase
# - task="summarization"           -> riassunto di un testo
#   es. testo lungo -> breve riassunto
sentiment_pipeline = pipeline(
    task="sentiment-analysis",
    model=model,
    tokenizer=tokenizer,
    device=device,
    truncation=True,
    max_length=128,
)

print("Modello caricato:", cfg.model_name)
print("Etichette modello:", model.config.id2label)

## 7. Funzioni modulari di inference, valutazione e alert

In [ ]:
# ============================================================
# FUNZIONI DI UTILITA'
# ============================================================

def normalize_model_label(label: str) -> str:
    """Converte eventuali label HuggingFace tipo LABEL_0 nei nomi sentiment."""
    label = label.lower()
    if label.startswith("label_"):
        label_id = int(label.replace("label_", ""))
        return cfg.label_map[label_id]
    return label

def predict_sentiment(texts: List[str], batch_size: int = 32, model_pipeline=None) -> Tuple[List[str], List[float]]:
    """Predice sentiment e confidence score per una lista di testi.

    Usa `sentiment_pipeline` (il modello originale) di default; passare
    `model_pipeline` per riusare la stessa logica di inferenza con un altro
    modello (es. la copia riaddestrata in sezione 12).
    """
    pipe = model_pipeline if model_pipeline is not None else sentiment_pipeline
    predictions = []
    scores = []

    for start in range(0, len(texts), batch_size):
        # prendo una porzione della lista texts
        # ["frase 0", "frase 1"]
        batch = texts[start:start + batch_size]
        outputs = pipe(batch, batch_size=batch_size)
        for out in outputs:
            if start in range(0, 10):
              print(out)
            predictions.append(normalize_model_label(out["label"]))
            scores.append(float(out["score"]))

    return predictions, scores

def evaluate_sentiment_model(df: pd.DataFrame, split_name: str) -> Dict[str, float]:
    """Valuta il modello su uno split e restituisce metriche principali."""
    y_true = df["sentiment"].tolist()
    y_pred = df["predicted_sentiment"].tolist()

    precision_macro, recall_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    precision_weighted, recall_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    return {
        "split": split_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision_macro": precision_macro,
        "recall_macro": recall_macro,
        "f1_macro": f1_macro,
        "precision_weighted": precision_weighted,
        "recall_weighted": recall_weighted,
        "f1_weighted": f1_weighted,
    }

def sample_for_test(df: pd.DataFrame, n: int, seed: int) -> pd.DataFrame:
    """Campiona lo split per rendere l'esecuzione sostenibile su Colab."""
    if len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=seed).reset_index(drop=True)

def evaluate_negative_share_alert(
    negative_share_by_week: pd.Series,
    n_current_weeks: int,
    max_share_increase: float,
) -> Dict[str, object]:
    """
    Confronta il periodo piu' recente con una baseline storica.
    LA BASELINE (media + 1 deviazione standard) viene calcolata SOLO sulle
    settimane precedenti al periodo corrente: usare l'intera serie (periodo
    corrente incluso) per costruire sia la baseline sia il valore da testare
    e' circolare e rende la soglia poco sensibile a un'anomalia reale.

    Vengono restituiti due segnali indipendenti, coerenti con le due soglie
    definite in ``Config``:
    - ``statistical_alert``: la quota del periodo corrente supera baseline + 1 std;
    - ``business_rule_alert``: l'incremento assoluto rispetto alla baseline
      supera ``max_negative_share_increase`` (soglia di business fissa).
    """
    weeks_list = negative_share_by_week.index.tolist()

    # Prendo il numero di settimane "correnti".
    # Se n_current_weeks è maggiore di len(weeks_list) - 1, prenderò len(weeks_list) - 1.
    # In questo ultimo caso il -1 evita che tutte vengano usate
    # come periodo corrente, perché deve rimanere almeno una settimana
    # da usare come baseline.
    n_current_weeks = min(n_current_weeks, len(weeks_list) - 1) if len(weeks_list) > 1 else 0

    # Quindi se ci sono settimane correnti (quasi sempre a true)
    if n_current_weeks:
        # identifico le settimane di baseline
        baseline_weeks = weeks_list[: len(weeks_list) - n_current_weeks]
        # identifico la/le settimana/e correnti
        current_weeks = weeks_list[len(weeks_list) - n_current_weeks:]
    else:
        baseline_weeks = weeks_list
        current_weeks = [weeks_list[-1]] if weeks_list else []

    # prende le settimane negative e controlla se fanno parte del periodo baseline_weeks
    baseline_series = negative_share_by_week.loc[baseline_weeks]
    baseline_mean = baseline_series.mean() # calcola la media
    # calcola la deviazione standard, cioè quanto i valori delle settimane precedenti 
    # tendono a variare rispetto alla loro media.
    # ddof=0 > divide per N
    baseline_std = baseline_series.std(ddof=0) 
    # Soglia base
    statistical_threshold = baseline_mean + baseline_std

    # prende le settimane negative e controlla se fanno parte del periodo corrente
    current_share = negative_share_by_week.loc[current_weeks].mean()
    share_increase = current_share - baseline_mean

    return {
        "baseline_weeks": baseline_weeks,
        "current_weeks": current_weeks,
        "baseline_mean": baseline_mean,
        "baseline_std": baseline_std,
        "statistical_threshold": statistical_threshold,
        "current_share": current_share,
        "share_increase": share_increase,
        "statistical_alert": bool(current_share > statistical_threshold),
        "business_rule_alert": bool(share_increase > max_share_increase),
    }


def build_monitoring_summary(
    macro_f1: float,
    avg_confidence: float,
    alert_result: Dict[str, object],
    rules: Dict[str, float],
) -> pd.DataFrame:
    """Costruisce la tabella di stato del monitoraggio (OK / RETRAINING_CANDIDATE /
    REVIEW_REQUIRED / REPUTATION_ALERT) a partire da metriche correnti e dal
    risultato di `evaluate_negative_share_alert`.

    Riusabile sia sui dati reali (sezione 12) sia su scenari sintetici, per
    dimostrare che ciascuno stato scatta davvero quando le condizioni si
    verificano, non solo su come si presentano i dati di questa esecuzione.
    """
    return pd.DataFrame([
        {
            "check": "Macro F1 minimo",
            "value": macro_f1,
            "threshold": rules["min_macro_f1"],
            "status": "OK" if macro_f1 >= rules["min_macro_f1"] else "RETRAINING_CANDIDATE",
        },
        {
            "check": "Confidence media minima",
            "value": avg_confidence,
            "threshold": rules["min_average_confidence"],
            "status": "OK" if avg_confidence >= rules["min_average_confidence"] else "REVIEW_REQUIRED",
        },
        {
            "check": "Quota sentiment negativo (soglia statistica)",
            "value": alert_result["current_share"],
            "threshold": alert_result["statistical_threshold"],
            "status": "OK" if not alert_result["statistical_alert"] else "REPUTATION_ALERT",
        },
        {
            "check": "Incremento sentiment negativo vs baseline",
            "value": alert_result["share_increase"],
            "threshold": rules["max_negative_share_increase"],
            "status": "OK" if not alert_result["business_rule_alert"] else "REPUTATION_ALERT",
        },
    ])

print("Funzioni caricate.")

## 8. Inference sul test set

Per mantenere il notebook veloce in Colab, la valutazione usa un campione del test set. In un ambiente di produzione useremmo l'intero test set o una suite di benchmark versionata.

In [ ]:
# ============================================================
# PREDIZIONE SUL TEST SET
# ============================================================

sample_test_df = sample_for_test(test_df, cfg.max_eval_samples, cfg.seed)

start_time = time.time()
preds, scores = predict_sentiment(sample_test_df["text"].tolist(), batch_size=cfg.batch_size)
elapsed = time.time() - start_time

sample_test_df["predicted_sentiment"] = preds
sample_test_df["confidence"] = scores

print(f"Esempi valutati: {len(sample_test_df):,}")
print(f"Tempo inference: {elapsed:.2f} secondi")
print(f"Tempo medio per testo: {elapsed / len(sample_test_df):.4f} secondi")

sample_test_df[["text", "sentiment", "predicted_sentiment", "confidence"]].head()

## 9. Valutazione delle performance

Usiamo piu' metriche per evitare una lettura troppo superficiale. L'accuracy e' intuitiva, ma con tre classi e possibile sbilanciamento e' importante guardare anche macro precision, macro recall e macro F1.

In [ ]:
# ============================================================
# METRICHE DI VALUTAZIONE
# ============================================================

metrics = evaluate_sentiment_model(sample_test_df, "test_sample")
metrics_df = pd.DataFrame([metrics]).set_index("split")
display(metrics_df.style.format("{:.4f}"))

print("Classification report:\n")
print(classification_report(
    sample_test_df["sentiment"],
    sample_test_df["predicted_sentiment"],
    labels=["negative", "neutral", "positive"],
    zero_division=0,
))

In [ ]:
# ============================================================
# GRAFICO 4: METRICHE PRINCIPALI
# ============================================================

metric_order = ["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]
plot_metrics = metrics_df.loc["test_sample", metric_order]

plt.figure(figsize=(10, 4))
sns.barplot(x=plot_metrics.index, y=plot_metrics.values, palette="viridis")
plt.ylim(0, 1)
plt.title("Metriche di performance sul campione di test")
plt.ylabel("Score")
plt.xlabel("Metrica")
plt.xticks(rotation=20, ha="right")
for i, val in enumerate(plot_metrics.values):
    plt.text(i, val + 0.02, f"{val:.3f}", ha="center", fontsize=9)
plt.tight_layout()
plt.show()

### Osservazione sulle metriche

Il modello raggiunge un'**accuracy del 70%** sul campione di test, con valori simili anche per le altre metriche principali:

- **Precision macro:** 0.699
- **Recall macro:** 0.710
- **F1-score macro:** 0.702
- **F1-score weighted:** 0.698

Le prestazioni risultano quindi **abbastanza bilanciate tra le classi**.

Analizzando i risultati per singola classe:

- **Negative** → è la classe riconosciuta meglio, con **F1-score = 0.73** e **recall = 0.79**.
- **Positive** → mostra prestazioni equilibrate, con **F1-score = 0.70**.
- **Neutral** → risulta la classe più difficile da identificare, con **recall = 0.63** e **F1-score = 0.67**.

Nel complesso, il modello mostra **prestazioni discrete e abbastanza uniformi**, ma presenta maggiore difficoltà nel riconoscimento dei testi **neutrali**.

In [ ]:
# ============================================================
# GRAFICO 5: MATRICE DI CONFUSIONE
# ============================================================

labels = ["negative", "neutral", "positive"]
cm = confusion_matrix(sample_test_df["sentiment"], sample_test_df["predicted_sentiment"], labels=labels)
cm_df = pd.DataFrame(cm, index=[f"Reale {l}" for l in labels], columns=[f"Pred {l}" for l in labels])

plt.figure(figsize=(7, 5))
sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False)
plt.title("Matrice di confusione - sentiment")
plt.tight_layout()
plt.show()

### Osservazione sulla matrice di confusione

La matrice di confusione permette di osservare **quali classi vengono riconosciute correttamente e quali vengono confuse tra loro**.

- **Negative:** 259 esempi su 328 vengono classificati correttamente. L'errore principale è la classificazione come **neutral** (63 casi).
- **Neutral:** 293 esempi su 462 sono corretti. È la classe che genera più errori: 110 esempi vengono classificati come **negative** e 59 come **positive**.
- **Positive:** 148 esempi su 210 vengono classificati correttamente. Gli errori sono soprattutto verso la classe **neutral** (53 casi), mentre solo 9 vengono classificati come **negative**.

Nel complesso, il modello distingue abbastanza bene i sentimenti **negative** e **positive**, mentre mostra maggiore difficoltà con la classe **neutral**, che tende a sovrapporsi alle altre due.

È inoltre interessante notare che la confusione diretta tra **positive** e **negative** è molto bassa (6 e 9 casi): gli errori avvengono soprattutto passando attraverso la classe **neutral**.

In [ ]:
# ============================================================
# GRAFICO 6: CONFIDENCE DEL MODELLO
# ============================================================

plt.figure(figsize=(10, 5))
sns.histplot(data=sample_test_df, x="confidence", hue="predicted_sentiment", bins=25, kde=True, palette="Set2")
plt.title("Distribuzione della confidence per sentiment predetto")
plt.xlabel("Confidence del modello")
plt.ylabel("Numero testi")
plt.tight_layout()
plt.show()

### Osservazione sulla confidence del modello

Il grafico mostra **quanto il modello è sicuro delle proprie predizioni**, distinguendo i risultati in base al sentiment predetto.

Per leggere il grafico:
- sull'**asse X** è riportata la **confidence**, cioè il livello di sicurezza associato alla previsione: valori più vicini a **1** indicano una maggiore sicurezza;
- sull'**asse Y** è riportato il **numero di testi** che presentano un determinato intervallo di confidence;
- i diversi **colori** rappresentano le tre classi predette: `negative`, `neutral` e `positive`;
- le **linee** aiutano a visualizzare l'andamento generale della distribuzione della confidence per ciascuna classe.

Dal grafico emerge che le predizioni **positive** sono spesso associate a confidence molto elevate, concentrate soprattutto verso **0.90–1.00**. Anche per la classe **negative** il modello mostra generalmente una buona sicurezza, con numerose predizioni ad alta confidence.

La classe **neutral** presenta invece una distribuzione più ampia, con molti valori anche tra circa **0.50 e 0.80**. Questo indica che il modello tende ad essere **meno sicuro quando assegna un testo alla classe neutral**.

Il risultato è coerente con le analisi precedenti: la classe `neutral` è quella che presenta il **recall e l'F1-score più bassi** e, nella matrice di confusione, è anche quella maggiormente confusa con le altre classi.

> **Nota:** una confidence elevata indica che il modello è molto sicuro della propria previsione, ma **non garantisce che la previsione sia corretta**. 

## 10. Instradamento a revisione umana (human-in-the-loop)

Come visto nell'osservazione precedente, una confidence alta non garantisce una previsione corretta, e la classe `neutral` è quella con confidence più dispersa e recall più basso. Ma il rischio non riguarda solo `neutral`: dalla matrice di confusione, un testo `negative` reale può finire predetto come `neutral` (63 casi) o come `positive` — quindi limitare la revisione ai soli testi predetti `negative` lascerebbe fuori proprio gli errori più pericolosi per il monitoraggio reputazionale: un negativo vero che il modello etichetta, magari con poca sicurezza, come `neutral` o `positive`, e che quindi non genererebbe mai un alert.

Per questo viene creata una **coda di revisione umana**: tutti i testi con confidence inferiore alla soglia `cfg.low_confidence_review_threshold` vengono
segnalati come casi più incerti e quindi candidati a un possibile controllo manuale.

La revisione **non esclude questi testi dal monitoraggio**: nel notebook tutte le predizioni vengono comunque utilizzate per calcolare l'andamento complessivo del sentiment.

In un sistema reale, il monitoraggio potrebbe utilizzare inizialmente la previsione del modello e, quando disponibile, sostituirla con l'**etichetta validata dal revisore umano**.

In [ ]:
# ============================================================
# CODA DI REVISIONE UMANA (HUMAN-IN-THE-LOOP)
# ============================================================
# Controlliamo la confidence su tutte e tre le classi predette.

review_queue_df = sample_test_df[
    sample_test_df["confidence"] < cfg.low_confidence_review_threshold
].sort_values("confidence")

print(f"Soglia di revisione: confidence < {cfg.low_confidence_review_threshold:.0%}")
print(f"Testi in coda di revisione: {len(review_queue_df):,} "
      f"su {len(sample_test_df):,} valutati "
      f"({len(review_queue_df) / len(sample_test_df):.2%} del campione).")
print("\nPer classe predetta:")
print(review_queue_df["predicted_sentiment"].value_counts().to_string())

review_queue_df[["text", "predicted_sentiment", "confidence"]].head(10)

### La soglia di confidence individua davvero i casi più a rischio?

Poiché nel test set conosciamo l'etichetta reale `sentiment`, possiamo verificare se i testi selezionati per la revisione contengono effettivamente **più errori rispetto al campione complessivo**.

Confrontiamo quindi:

- il **tasso di errore complessivo** del modello;
- il **tasso di errore nella coda di revisione**, composta dai testi con confidence bassa.

Se il secondo valore è maggiore, significa che la soglia sta effettivamente concentrando nella coda i casi più problematici.

In [ ]:
# ============================================================
# VALIDAZIONE DELLA SOGLIA DI CONFIDENCE
# (possibile solo qui, dove abbiamo la ground truth del benchmark)
# ============================================================
# Confrontiamo predicted_sentiment con la vera etichetta "sentiment", che in
# produzione non avremo mai per dati nuovi. Non stiamo "correggendo" nulla:
# stiamo verificando se la soglia di bassa confidence individua davvero i
# casi sbagliati, prima di doverci fidare solo di quella in produzione.

review_queue_error_rate = (review_queue_df["predicted_sentiment"] != review_queue_df["sentiment"]).mean()
overall_error_rate = (sample_test_df["predicted_sentiment"] != sample_test_df["sentiment"]).mean()

print(f"Tasso di errore nella coda di revisione (bassa confidence): {review_queue_error_rate:.2%}")
print(f"Tasso di errore medio sull'intero campione:                 {overall_error_rate:.2%}")
print(f"La coda di revisione concentra un tasso di errore "
      f"{review_queue_error_rate / overall_error_rate:.1f}x piu' alto della media del campione.")

### Come leggere questo risultato

Sul campione completo il modello sbaglia circa **3 testi su 10**.

Se invece guardiamo solo i testi con **confidence bassa**, cioè quelli che il modello considera più incerti, il tasso di errore sale a circa **1 testo su 2**.

Questo significa che la soglia di confidence è utile: i testi con confidence bassa sono effettivamente **più rischiosi** e quindi ha senso mandarli in revisione. **Attenzione però:** una confidence bassa non significa che la previsione sia sicuramente sbagliata. Molti testi con confidence bassa sono comunque classificati correttamente.

La coda di revisione va quindi interpretata come:

> **insieme di casi più incerti e quindi da controllare**, non come insieme di errori certi.


### Dalla revisione al fine-tuning

Nel test set possiamo verificare automaticamente se una previsione è corretta
perché conosciamo l'etichetta reale `sentiment`.

In produzione, invece, sui nuovi testi questa etichetta non sarebbe disponibile.
Per questo i casi con **confidence bassa**, già identificati nella coda di
revisione, potrebbero essere controllati da un revisore umano.

Il processo sarebbe:

1. I testi incerti vengono salvati, ad esempio, in un file CSV.
2. Un revisore umano legge i testi e assegna l'etichetta corretta.
3. Le correzioni vengono conservate.
4. Nel tempo queste correzioni possono formare un nuovo dataset etichettato.
5. Il nuovo dataset può essere utilizzato per un eventuale **fine-tuning** del modello.

Il ciclo diventa quindi:

**il modello predice → i casi incerti vengono revisionati → l'umano corregge → le correzioni possono essere usate per migliorare il modello**

## 11. Monitoraggio continuo della reputazione

Finora abbiamo valutato il modello su un campione del test set, ottenendo una fotografia delle sue prestazioni in un determinato momento.

In un'applicazione reale di **sentiment analysis**, però, può essere utile monitorare anche **come cambia il sentiment nel tempo**.

Ad esempio, un'azienda potrebbe voler sapere:

> **La percentuale di commenti negativi sta aumentando rispetto alle settimane precedenti?**

Un sistema reale potrebbe ricevere continuamente nuovi testi da **social network, recensioni, API o strumenti di social listening**, classificarli con il modello e aggregare i risultati nel tempo.

In questa sezione simuliamo questo processo in due passaggi:

1. **Monitoraggio temporale** → calcoliamo la percentuale di sentiment `negative`, `neutral` e `positive` per ogni settimana.
2. **Sistema di alert** → controlliamo se la quota di sentiment negativo aumenta in modo anomalo rispetto alle settimane precedenti.

> **Nota:** il dataset utilizzato non contiene una vera dimensione temporale.  
> Le date vengono quindi generate artificialmente solo per simulare il funzionamento di un sistema di monitoraggio reale.

### 11.1 Simulazione del monitoraggio temporale

Per simulare l'arrivo continuo di nuovi testi, partiamo dalle predizioni effettuate sul test set e assegniamo a ogni testo una **data fittizia**.

Successivamente:

- raggruppiamo i testi per **settimana**;
- contiamo quanti testi sono stati classificati come `negative`, `neutral` e `positive`;
- trasformiamo i conteggi in **quote percentuali**;
- visualizziamo l'andamento delle tre classi nel tempo.

In questo modo possiamo simulare una semplice **dashboard di monitoraggio della reputazione**.

In [ ]:
# ============================================================
# SIMULAZIONE DEL MONITORAGGIO TEMPORALE
# ============================================================

# Viene creata una copia del DataFrame contenente le predizioni
# per non modificare direttamente sample_test_df.
monitor_df = sample_test_df.copy()

# ------------------------------------------------------------
# 1. SIMULAZIONE DELLE DATE
# ------------------------------------------------------------
# Il dataset non contiene date reali.
# Viene assegnata quindi artificialmente una data a ogni testo. 
# L'ultima settimana della simulazione coincide sempre con
# la settimana corrente reale, qualunque sia il giorno in cui si esegue il
# notebook. 
# ES. DATE: 10/09/26 12:16:00
# monitor_df["date"] = 
# - riga 0 > 2026-07-30 19:00:00
# - riga 1 > 2026-07-30 20:00:00
# - riga 2 > 2026-07-30 21:00:00

# Crea tante date quante sono le righe di monitor_df, distanziate di un'ora, 
# facendo terminare la serie all'ora corrente.
monitor_df["date"] = pd.date_range(
    # data e ora attuale e .floor("h") arrotonda per difetto all'ora intera
    end=pd.Timestamp.now().floor("h"),
    periods=len(monitor_df),
    freq="h" # crea una data ogni ora
)
# Conversione di ogni data nella settimana corrispondente.
# monitor_df["week"] = 
# date                 week
# 2026-07-30 19:00     2026-07-27/2026-08-02
# 2026-07-30 20:00     2026-07-27/2026-08-02
# 2026-07-30 21:00     2026-07-27/2026-08-02
monitor_df["week"] = monitor_df["date"].dt.to_period("W").astype(str)

In [ ]:
# ------------------------------------------------------------
# 2. CONTEGGIO DEI SENTIMENT PER SETTIMANA
# ------------------------------------------------------------
sentiment_order = ["negative", "neutral", "positive"]

# Per ogni settimana vengono quanti testi sono stati classificati
# come negative, neutral e positive.
# Esempio del risultato atteso:
# week                    negative   neutral   positive
# 2026-07-27/2026-08-02      30        50        20
# 2026-08-03/2026-08-09      25        55        20
#
# unstack() trasforma le classi di sentiment in colonne.
# fill_value=0 assegna 0 se in una settimana manca una classe.
weekly_counts = (
    monitor_df
    .groupby(["week", "predicted_sentiment"])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=sentiment_order, fill_value=0)
    .sort_index()
)
print(weekly_counts)

# ------------------------------------------------------------
# 3. DAL NUMERO DI TESTI ALLA QUOTA DI SENTIMENT
# ------------------------------------------------------------
# Conversione dei conteggi in proporzioni.
# Esempio:
# 30 negative su 100 testi -> quota negative = 0.30
# In ogni settimana:
# negative + neutral + positive = 1
weekly_shares = weekly_counts.div(
    weekly_counts.sum(axis=1),
    axis=0
)

In [ ]:
weekly_shares

In [ ]:
# ------------------------------------------------------------
# 4. PREPARAZIONE DEI DATI PER IL GRAFICO
# ------------------------------------------------------------
# Seaborn lavora più comodamente con una riga per ogni
# combinazione settimana/sentiment.
# Viene trasformato quindi:
# week       negative   neutral   positive
# settimana1   0.30      0.50      0.20
# in:
# week                        predicted_sentiment    share
# 2026-07-27/2026-08-02       negative               0.30
# 2026-08-03/2026-08-09       negative               0.50
# 2026-08-10/2026-08-16       negative               0.20
# 2026-07-27/2026-08-02       neutral                0.30
# 2026-08-03/2026-08-09       neutral                0.50
# 2026-08-10/2026-08-16       neutral                0.20
# 2026-07-27/2026-08-02       positive               0.30
# 2026-08-03/2026-08-09       positive               0.50
# 2026-08-10/2026-08-16       positive               0.20
weekly_sentiment = (
    weekly_shares
    .reset_index()
    .melt(
        id_vars="week",
        value_vars=sentiment_order,
        var_name="predicted_sentiment",
        value_name="share"
    )
)
print(weekly_sentiment)

# ------------------------------------------------------------
# 5. GRAFICO DELL'ANDAMENTO DEL SENTIMENT
# ------------------------------------------------------------
plt.figure(figsize=(12, 5))
sns.lineplot(
    data=weekly_sentiment,
    x="week",
    y="share",
    hue="predicted_sentiment",
    marker="o"
)
plt.title("Monitoraggio settimanale del sentiment predetto")
plt.xlabel("Settimana")
plt.ylabel("Quota di sentiment")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Osservazione sul monitoraggio settimanale

Il grafico mostra come cambia, settimana dopo settimana, la **quota di testi classificati come `negative`, `neutral` e `positive`**.

Si possono osservare alcune variazioni tra le settimane. In particolare, il sentiment `negative` si mantiene generalmente tra circa il **35% e il 42%**, mentre il sentiment `neutral` presenta variazioni più evidenti, raggiungendo il valore più alto nella settimana **24/08–30/08**. Il sentiment `positive` rimane invece generalmente meno frequente rispetto alle altre due classi.

Questo tipo di monitoraggio permette di osservare nel tempo la distribuzione del sentiment e di individuare eventuali **aumenti della quota di commenti negativi**, che potrebbero richiedere maggiore attenzione.

### 11.2 Alert sul sentiment negativo

Il grafico permette di osservare manualmente l'andamento della reputazione, ma in un sistema reale non possiamo controllarlo continuamente.

Introduciamo quindi un **sistema automatico di alert**.

L'idea è confrontare la quota di sentiment negativo del **periodo corrente** con il comportamento osservato nelle settimane precedenti, utilizzate come **baseline**.

La baseline rappresenta quindi il comportamento considerato "normale" del sistema.

Il controllo utilizza due criteri:

1. **Alert statistico**  
   Verifica se la quota negativa corrente è insolitamente alta rispetto alla media e alla variabilità osservate nella baseline.

2. **Alert di business**  
   Verifica se l'aumento della quota negativa supera una soglia massima definita a priori.

Ad esempio:

> Se normalmente circa il 20% dei testi è negativo e improvvisamente la quota sale al 40%, il sistema può segnalare un possibile peggioramento della reputazione.

In [ ]:
# ============================================================
# ALERT SUL SENTIMENT NEGATIVO: BASELINE STORICA VS PERIODO CORRENTE
# ============================================================

negative_df_prop_by_week = weekly_shares["negative"]

alert_result = evaluate_negative_share_alert(
    negative_df_prop_by_week,
    n_current_weeks=cfg.n_current_weeks_for_monitoring,
    max_share_increase=cfg.max_negative_share_increase,
)

In [ ]:
# ============================================================
# RISULTATI DELL'ALERT
# ============================================================
print(f"Settimane di baseline: {alert_result['baseline_weeks']}")
print(f"Settimana/e corrente/i: {alert_result['current_weeks']}")
print()

print(f"Quota negativa media nella baseline: "
      f"{alert_result['baseline_mean']:.2%}")
print(f"Soglia statistica (baseline + 1 std): "
      f"{alert_result['statistical_threshold']:.2%}")
print(f"Quota negativa nel periodo corrente: "
      f"{alert_result['current_share']:.2%}")
print(f"Incremento rispetto alla baseline: "
      f"{alert_result['share_increase']:+.2%}")
print()

print(f"Alert statistico: "
      f"{alert_result['statistical_alert']}")
print(
    f"Alert regola di business "
    f"(incremento > {cfg.max_negative_share_increase:.0%}): "
    f"{alert_result['business_rule_alert']}"
)

# ============================================================
# TABELLA DI CONTROLLO
# ============================================================
alert_df = negative_df_prop_by_week.reset_index(name="negative_share")
alert_df["is_current_period"] = alert_df["week"].isin(alert_result["current_weeks"])
alert_df["exceeds_statistical_threshold"] = alert_df["negative_share"] > alert_result["statistical_threshold"]
display(alert_df)

### Osservazione sugli alert

L'obiettivo di questo controllo è capire se, nell'**ultima settimana**, la percentuale di testi classificati come `negative` è aumentata abbastanza da richiedere attenzione.

#### Come leggere il risultato

Le settimane precedenti vengono utilizzate come **baseline**, cioè come riferimento per capire quale sia il livello abituale di sentiment negativo.

Nel nostro caso:

- **Media della baseline:** `37,28%` 
  → nelle settimane precedenti a quella attuale, in media, il 37,28% dei testi era classificato come negativo.

- **Quota negativa della settimana corrente:** `42,19%`  
  → nella settimana corrente il 42,19% dei testi è stato classificato come negativo.

- **Incremento rispetto alla baseline:** `+4,91 punti percentuali`  
  → la quota di sentiment negativo è quindi aumentata rispetto alla media delle settimane precedenti.

A questo punto vengono effettuati **due controlli differenti**.

#### 1. Alert statistico → `True`

La soglia statistica, ovvero la soglia oltre la quale la percentuale è considerata insolita, è `41,37%`.

La settimana corrente raggiunge `42,19%` e quindi **supera questa soglia**.

Il sistema segnala quindi che il livello di sentiment negativo dell'ultima settimana è **più alto rispetto a quanto atteso sulla base delle settimane precedenti**.

#### 2. Alert di business → `False`

La regola di business richiede invece un aumento superiore a **15 punti percentuali** rispetto alla media della baseline.

L'aumento osservato è solamente di **+4,91 punti percentuali**, quindi questa soglia non viene superata e l'alert di business non scatta.

In questo caso scatta solo l'alert statistico (borderline, +4,91 punti percentuali), non quello di business: non e' una crisi reputazionale netta. La sezione 12.1 verifica il meccanismo su uno scenario sintetico con un aumento piu' marcato, in cui anche l'alert di business scatta.


## 12. Monitoraggio del modello

Una volta messo in produzione, un modello deve essere monitorato nel tempo per verificare che continui a fornire predizioni affidabili.

Il **retraining non viene eseguito automaticamente**, ma può essere preso in considerazione quando alcuni indicatori mostrano un possibile peggioramento delle prestazioni del modello.

In questa sezione definiamo quindi alcune semplici regole di monitoraggio:

- il **Macro F1** viene confrontato con una soglia minima;
- la **confidence media** viene confrontata con una soglia minima;
- vengono inoltre riportati gli alert sul sentiment negativo calcolati nella sezione precedente.

Gli alert sulla reputazione hanno però uno scopo diverso: segnalano un possibile aumento dei commenti negativi, ma **non indicano direttamente la necessità di effettuare il retraining del modello**.

In [ ]:
# ============================================================
# REGOLE DI MONITORAGGIO PER IL RETRAINING
# ============================================================

monitoring_rules = {
    "min_macro_f1": cfg.min_macro_f1,
    "max_negative_share_increase": cfg.max_negative_share_increase,
    "min_average_confidence": cfg.min_average_confidence,
}

current_macro_f1 = metrics["f1_macro"]
current_avg_confidence = sample_test_df["confidence"].mean()

monitoring_summary = build_monitoring_summary(
    macro_f1=current_macro_f1,
    avg_confidence=current_avg_confidence,
    alert_result=alert_result,
    rules=monitoring_rules,
)

display(monitoring_summary)


### 12.1 Cosa succede quando scattano i tre stati?

Nel monitoraggio reale mostrato sopra, il Macro F1 e la confidence media rientrano nelle soglie. La quota di sentiment negativo, invece, supera gia' la soglia statistica (non quella di business, si veda l'osservazione in sezione 11.2): il monitoraggio genera quindi un primo REPUTATION_ALERT reale, non solo ipotetico. Per vedere anche gli altri due stati in azione, e un caso di REPUTATION_ALERT piu' netto, costruiamo tre semplici **scenari sintetici**.

Ogni scenario modifica un solo valore alla volta, mantenendo gli altri in condizioni normali. In questo modo possiamo vedere chiaramente quale controllo provoca ciascuno stato:

- `RETRAINING_CANDIDATE` → il **Macro F1** scende sotto la soglia minima: le prestazioni del modello potrebbero essere peggiorate e può essere necessario valutare un nuovo training.
- `REVIEW_REQUIRED` → la **confidence media** scende sotto la soglia minima: il modello sta producendo predizioni mediamente meno sicure e può essere utile aumentare la revisione umana.
- `REPUTATION_ALERT` → la **quota di sentiment negativo** aumenta oltre le soglie definite: viene segnalato un possibile peggioramento della reputazione da approfondire.

> I valori utilizzati nei tre scenari sono artificiali e servono esclusivamente a verificare il comportamento delle regole di monitoraggio.

In [ ]:
# ============================================================
# 12.1 DIMOSTRAZIONE DEI TRE POSSIBILI STATI
# ============================================================

# Situazione normale della reputazione:
# nessuna delle due soglie relative al sentiment negativo viene superata.
# La utilizziamo nei primi due scenari per isolare il problema
# rispettivamente sul Macro F1 e sulla confidence.
normal_alert_result = {
    "current_share": 0.19,
    "statistical_threshold": 0.25,
    "statistical_alert": False,
    "share_increase": 0.00,
    "business_rule_alert": False,
}

In [ ]:
# ------------------------------------------------------------
# SCENARIO 1: Macro F1 troppo basso
# ------------------------------------------------------------
# Simuliamo un modello con F1 = 0.55, inferiore alla soglia di 0.70.
# Confidence e reputazione rimangono invece nella norma.
scenario_retraining = build_monitoring_summary(
    macro_f1=0.55,
    avg_confidence=0.80,
    alert_result=normal_alert_result,
    rules=monitoring_rules,
)

scenario_retraining.insert(
    0,
    "scenario",
    "F1 macro basso (RETRAINING_CANDIDATE)"
)

In [ ]:
# ------------------------------------------------------------
# SCENARIO 2: Confidence media troppo bassa
# ------------------------------------------------------------
# Il Macro F1 rimane buono, ma la confidence media scende a 0.45,
# sotto la soglia minima di 0.60.
scenario_review = build_monitoring_summary(
    macro_f1=0.80,
    avg_confidence=0.45,
    alert_result=normal_alert_result,
    rules=monitoring_rules,
)

scenario_review.insert(
    0,
    "scenario",
    "Confidence media bassa (REVIEW_REQUIRED)"
)

In [ ]:
# ------------------------------------------------------------
# SCENARIO 3: aumento evidente del sentiment negativo
# ------------------------------------------------------------

# Simuliamo cinque settimane con una quota negativa abbastanza stabile
# e un forte aumento nell'ultima settimana.
demo_negative_share_by_week = pd.Series(
    {
        "2026-W01": 0.18,
        "2026-W02": 0.20,
        "2026-W03": 0.17,
        "2026-W04": 0.19,
        "2026-W05": 0.21,
        "2026-W06": 0.42,
    },
    name="negative_share",
)

# Applichiamo la stessa funzione utilizzata nel monitoraggio reale.
# In questo caso l'aumento è costruito appositamente
# per superare sia la soglia statistica sia quella di business.
reputation_alert_result = evaluate_negative_share_alert(
    demo_negative_share_by_week,
    n_current_weeks=cfg.n_current_weeks_for_monitoring,
    max_share_increase=cfg.max_negative_share_increase,
)

# Macro F1 e confidence rimangono nella norma:
# l'unico problema dello scenario riguarda la reputazione.
scenario_reputation = build_monitoring_summary(
    macro_f1=0.80,
    avg_confidence=0.80,
    alert_result=reputation_alert_result,
    rules=monitoring_rules,
)

scenario_reputation.insert(
    0,
    "scenario",
    "Picco di sentiment negativo (REPUTATION_ALERT)"
)

In [ ]:
reputation_alert_result

In [ ]:
# ------------------------------------------------------------
# UNIONE DEI TRE SCENARI
# ------------------------------------------------------------

three_scenarios_df = pd.concat(
    [
        scenario_retraining,
        scenario_review,
        scenario_reputation,
    ],
    ignore_index=True,
)

display(three_scenarios_df)

### Osservazione sui risultati

I risultati confermano il comportamento atteso nei tre scenari simulati.

- Nel **primo scenario**, il Macro F1 è `0.55`, inferiore alla soglia minima di `0.70`: viene quindi restituito `RETRAINING_CANDIDATE`. Gli altri controlli rimangono `OK`.
- Nel **secondo scenario**, la confidence media è `0.45`, inferiore alla soglia minima di `0.60`: viene restituito `REVIEW_REQUIRED`, mentre gli altri controlli rimangono `OK`.
- Nel **terzo scenario**, la quota di sentiment negativo raggiunge il `42%`, superando la soglia statistica di circa `20.41%`. Anche l'incremento rispetto alla baseline, pari a `23 punti percentuali`, supera la soglia di business di `15 punti percentuali`: entrambi i controlli generano quindi `REPUTATION_ALERT`.

La simulazione mostra quindi che il sistema riesce a **distinguere correttamente le tre situazioni** e ad attivare solamente gli stati associati alle soglie che vengono superate.

> I valori utilizzati sono stati costruiti artificialmente per testare le regole di monitoraggio e non rappresentano risultati reali del modello.

## 12.2 Cosa fare quando scatta uno dei tre stati?

Dopo aver verificato che i tre stati vengono riconosciuti correttamente, possiamo associare ad ognuno una possibile azione.

- `RETRAINING_CANDIDATE` → preparare un dataset aggiornato e valutare un nuovo training del modello.
- `REVIEW_REQUIRED` → inviare i casi meno sicuri alla revisione umana.
- `REPUTATION_ALERT` → analizzare i testi negativi e capire cosa sta causando l'aumento.

Le celle seguenti mostrano una possibile gestione dei tre casi.

Ogni azione viene eseguita solo se lo stato reale calcolato sopra (`monitoring_summary`) lo richiede davvero. Per le azioni che con i dati di questa esecuzione non scatterebbero, un flag esplicito (`FORCE_..._DEMO`) forza comunque la dimostrazione, sullo stesso principio gia' usato per la demo Gradio in sezione 14 (`RUN_GRADIO_DEMO`): in produzione quel flag andrebbe rimosso, lasciando decidere solo il monitoraggio reale.


### Scenario 1 — `RETRAINING_CANDIDATE`: retraining del modello

Se il Macro F1 scende sotto la soglia minima, il modello viene considerato candidato al retraining.

Prima di riaddestrarlo bisogna scegliere quali dati utilizzare. Se sono disponibili correzioni effettuate dai revisori umani, vengono preferite perché rappresentano nuovi esempi già validati. In assenza di correzioni umane, nel notebook viene utilizzato un campione del training set originale per dimostrare il processo.

In [ ]:
# ============================================================
# 12.3 SELEZIONE DEL DATASET DI RETRAINING
# ============================================================

def get_retraining_dataset(n_default: int = 300, seed: int = cfg.seed) -> pd.DataFrame:
    """Dataset di correzioni umane se esiste, altrimenti fallback a train_df (vedi cella markdown sopra)."""
    if "human_corrected_df" in globals() and len(human_corrected_df) > 0:
        print(f"Uso il dataset di correzioni umane: {len(human_corrected_df):,} esempi.")
        return human_corrected_df[["text", "sentiment"]].copy()

    print("Nessun dataset di correzioni umane disponibile (human_corrected_df non definito): "
          "uso un campione di default da train_df, per dimostrazione.")

    # calcola quanti esempi prendere per ogni classe.
    # n_default viene diviso per il numero di classi presenti in cfg.label_map.
    # max(1, ...) garantisce che venga preso almeno un esempio per classe.  
    n_example_per_class = max(1, n_default // len(cfg.label_map))
    return (
        train_df
        .groupby("sentiment", group_keys=False) # divide il DataFrame in gruppi in base alla colonna sentiment
        # per ogni gruppo:
        # - estrae casualmente alcuni esempi (g.sample)
        # - decide quanti esempi prendere
        .apply(lambda g: g.sample(n=min(len(g), n_example_per_class), random_state=seed))
        [["text", "sentiment"]]
        .reset_index(drop=True) # ricrea l'indice da zero. 
        # drop=True evita che il vecchio indice venga salvato come nuova colonna
    )

In [ ]:
# ============================================================
# GATE: il retraining parte solo se il monitoraggio reale lo richiede
# ============================================================
run_retraining_demo = (monitoring_summary["status"] == "RETRAINING_CANDIDATE").any()

print(f"Serve il retraining secondo il monitoraggio reale? {run_retraining_demo}")

if run_retraining_demo:
    retraining_df = get_retraining_dataset()
    print(f"\nDataset di retraining selezionato: {len(retraining_df):,} esempi.")
    print(retraining_df["sentiment"].value_counts().to_string())
else:
    print("Nessun retraining necessario: il Macro F1 resta entro la soglia minima.")


In [ ]:
# ============================================================
# 12.4 RETRAINING DIMOSTRATIVO
# ============================================================

if run_retraining_demo:
    from datasets import Dataset as HFDataset
    from transformers import DataCollatorWithPadding, Trainer, TrainingArguments

    label2id = {name: idx for idx, name in cfg.label_map.items()}

    def _tokenize_batch(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    retrain_hf_dataset = HFDataset.from_pandas(retraining_df[["text", "sentiment"]])
    retrain_hf_dataset = retrain_hf_dataset.map(
        lambda batch: {"label": [label2id[s] for s in batch["sentiment"]]},
        batched=True,
    )
    retrain_hf_dataset = retrain_hf_dataset.map(_tokenize_batch, batched=True)
    retrain_hf_dataset = retrain_hf_dataset.remove_columns(
        [c for c in retrain_hf_dataset.column_names if c not in ("input_ids", "attention_mask", "label")]
    )

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

    # Copia indipendente (vedi cella markdown sopra)
    retrain_model = AutoModelForSequenceClassification.from_pretrained(cfg.model_name)

    training_args = TrainingArguments(
        output_dir="./retraining_demo_output",
        num_train_epochs=1,
        per_device_train_batch_size=8,
        learning_rate=2e-5,
        logging_steps=10,
        save_strategy="no",
        report_to=[],
    )

    trainer = Trainer(
        model=retrain_model,
        args=training_args,
        train_dataset=retrain_hf_dataset,
        data_collator=data_collator,
    )

    print(f"Retraining dimostrativo su {len(retrain_hf_dataset):,} esempi, "
          f"{training_args.num_train_epochs} epoca/e...")
    trainer.train()


In [ ]:
# ============================================================
# 12.5 CONFRONTO PRIMA / DOPO IL RETRAINING DIMOSTRATIVO
# ============================================================
# Rivalutiamo sullo stesso campione di test (sample_test_df) con il modello
# appena riaddestrato, e confrontiamo con le metriche originali (sezione 9,
# calcolate prima di qualunque retraining).

if run_retraining_demo:
    retrain_pipeline = pipeline(
        task="sentiment-analysis",
        model=retrain_model,
        tokenizer=tokenizer,
        device=device,
        truncation=True,
        max_length=128,
    )

    preds_after, scores_after = predict_sentiment(
        sample_test_df["text"].tolist(),
        batch_size=cfg.batch_size,
        model_pipeline=retrain_pipeline,
    )

    after_df = sample_test_df.copy()
    after_df["predicted_sentiment"] = preds_after
    after_df["confidence"] = scores_after

    metrics_after = evaluate_sentiment_model(after_df, "dopo_retraining_demo")

    comparison_df = pd.DataFrame([metrics, metrics_after]).set_index("split")
    comparison_df.index = ["prima (modello originale)", "dopo (fine-tuning dimostrativo)"]
    display(comparison_df[["accuracy", "precision_macro", "recall_macro", "f1_macro", "f1_weighted"]].style.format("{:.4f}"))
else:
    print("Retraining non eseguito: nessun confronto prima/dopo da mostrare.")


**Come leggere il confronto**

*(la tabella sopra compare solo se `run_retraining_demo` e' `True`)*

Confronta le due righe: se le metriche `dopo` sono vicine o leggermente migliori di quelle `prima`, il fine-tuning dimostrativo ha fatto il suo lavoro senza rompere nulla — non e' un risultato scontato, con cosi' pochi esempi e' altrettanto probabile che il modello *peggiori* leggermente (overfitting su un campione piccolo) quanto che migliori in modo significativo. Se il punteggio `dopo` fosse sensibilmente piu' basso, sarebbe un segnale che il dataset di retraining usato (`retraining_df`) era troppo piccolo o poco rappresentativo, o che servirebbero piu' epoche/un learning rate diverso — esattamente il tipo di iterazione e validazione che un vero processo di retraining in produzione richiederebbe prima di sostituire il modello in uso.


### Scenario 2 — `REVIEW_REQUIRED`: revisione umana

Se la confidence media scende sotto la soglia minima la prima azione consiste nell'individuare le predizioni meno sicure e inviarle alla revisione umana.

In [ ]:
# ============================================================
# AZIONE: REVIEW_REQUIRED -> instradamento a revisione umana
# ============================================================
# I dati da mostrare esistono gia': e' la coda di revisione costruita in
# sezione 10 (review_queue_df), non serve ricalcolare nulla qui.
needs_review = (monitoring_summary["status"] == "REVIEW_REQUIRED").any()

print(f"Serve la revisione secondo il monitoraggio reale? {needs_review}")

if needs_review:
    print(f"\nAzione: instradare a revisione umana i {len(review_queue_df):,} "
          f"testi gia' individuati in sezione 10 "
          f"(confidence < {cfg.low_confidence_review_threshold:.0%}).")
    display(review_queue_df[["text", "predicted_sentiment", "confidence"]].head(10))
else:
    print("Nessuna azione necessaria: la confidence media resta sopra soglia.")


### Scenario 3 — `REPUTATION_ALERT`: analisi del sentiment negativo

Se aumenta in modo anomalo la quota di sentiment negativo, il problema non riguarda necessariamente il modello.

In questo caso bisogna analizzare i testi classificati come negativi per capire quali argomenti stanno causando l'aumento.

In [ ]:
# ============================================================
# AZIONE: REPUTATION_ALERT -> analisi dei testi negativi del periodo corrente
# ============================================================
# I dati da mostrare esistono gia': sono i testi del monitoraggio settimanale
# (sezione 11.1, monitor_df) nella/e settimana/e corrente/i individuata/e
# dall'alert reale (sezione 11.2, alert_result).
needs_reputation_check = (monitoring_summary["status"] == "REPUTATION_ALERT").any()

print(f"Serve approfondire la reputazione secondo il monitoraggio reale? "
      f"{needs_reputation_check}")

if needs_reputation_check:
    current_negative_texts = monitor_df[
        monitor_df["week"].isin(alert_result["current_weeks"])
        & (monitor_df["predicted_sentiment"] == "negative")
    ]
    print(f"\nAzione: analizzare i {len(current_negative_texts):,} testi negativi "
          f"della settimana corrente per capire cosa sta causando l'aumento.")
    display(current_negative_texts[["week", "text", "confidence"]].head(10))
else:
    print("Nessuna azione necessaria: il sentiment negativo resta entro le soglie.")


## 13. Pipeline CI/CD proposta

La consegna richiede una repository GitHub con codice documentato per pipeline CI/CD e implementazione. I file applicativi (predictor, demo, test) vivono nella cartella `sentiment_reputation_mlops/` (vedi struttura sotto). Il workflow `ci.yml` invece sta alla radice del repository GitHub, non dentro questa cartella: GitHub Actions legge i workflow solo da `.github/workflows/` nella vera radice del repository, mai da una sottocartella. Un filtro `paths` nel workflow lo fa comunque scattare solo quando cambia qualcosa dentro `sentiment_reputation_mlops/`.

In [ ]:
# ============================================================
# STRUTTURA DEL REPOSITORY (riferimento — i file vivono nel repo, non qui)
# ============================================================

REPO_STRUCTURE = """
<radice del repository GitHub>
├── .github/
│   └── workflows/
│       └── ci.yml                # installa le dipendenze ed esegue pytest ad ogni push/PR su main
│                                 # (filtrato ai soli cambi dentro sentiment_reputation_mlops/)
└── sentiment_reputation_mlops/
    ├── requirements.txt          # dipendenze del repository (transformers, gradio, ecc.)
    ├── predictor.py              # SentimentPredictor: carica il modello una volta, espone predict()
    ├── app.py                    # demo Gradio, usa SentimentPredictor da predictor.py
    ├── conftest.py               # vuoto: serve solo perche' pytest trovi predictor.py da tests/
    └── tests/
        └── test_smoke.py         # test_model_loads + test_known_examples, usano SentimentPredictor
"""

print(REPO_STRUCTURE)


**Osservazione sulla pipeline CI/CD**

`app.py` e `tests/test_smoke.py` importano `SentimentPredictor` da `predictor.py`, che accentra il caricamento e la normalizzazione dell'output in un solo posto.

`test_known_examples` verifica che due frasi non ambigue vengano classificate nella classe attesa — un primo, piccolo test di regressione sulla qualita', non solo sul "si avvia senza errori". Il file `conftest.py` (vuoto) e' un dettaglio tecnico non ovvio ma necessario: senza di esso, pytest aggiungerebbe a `sys.path` solo la cartella `tests/`, e `from predictor import SentimentPredictor` in `test_smoke.py` fallirebbe con `ModuleNotFoundError` perche' non troverebbe `predictor.py` nella radice del repository.

In un progetto reale aggiungerei anche controlli sul formato dei dati, linting e deploy automatico su HuggingFace Spaces solo dopo il superamento dei test. Il punto importante resta che il modello non dovrebbe arrivare in produzione solo perche' il notebook funziona: serve una catena automatica che controlli codice, dipendenze e comportamento minimo del sistema.

## 14. Demo facoltativa con Gradio

Questa demo permette di provare il modello su testi inseriti manualmente. In Colab puo' essere avviata impostando `RUN_GRADIO_DEMO = True`.

In [ ]:
# ============================================================
# DEMO GRADIO FACOLTATIVA
# ============================================================

RUN_GRADIO_DEMO = False

if RUN_GRADIO_DEMO:
    import gradio as gr

    def classify_text(text: str):
        result = sentiment_pipeline(text)[0]
        return {"sentiment": result["label"], "confidence": round(float(result["score"]), 4)}

    demo = gr.Interface(
        fn=classify_text,
        inputs=gr.Textbox(lines=4, label="Testo social"),
        outputs=gr.JSON(label="Risultato"),
        title="Monitoraggio reputazione online - Sentiment Analysis",
        description="Demo del modello CardiffNLP per classificare testi social in negative, neutral o positive.",
    )
    demo.launch(share=True)
else:
    print("Demo Gradio non avviata. Impostare RUN_GRADIO_DEMO = True per provarla.")

## 15. Conclusioni finali

Il progetto mostra come costruire una base completa per il monitoraggio della reputazione online tramite sentiment analysis. Il modello CardiffNLP permette di partire da una soluzione gia' addestrata su testi social, quindi adatta al linguaggio breve e rumoroso tipico di Twitter.

La parte piu' importante non e' solo la classificazione del singolo testo, ma il flusso complessivo: valutare le performance, controllare dove il modello sbaglia, instradare i casi incerti a revisione umana, monitorare l'andamento del sentiment nel tempo e definire regole per alert e retraining.

Dal punto di vista MLOps, il notebook propone una struttura replicabile:

- dataset pubblico e versionabile;
- modello pre-addestrato con riferimento HuggingFace corretto;
- configurazione centralizzata, comprese le soglie operative di monitoraggio e revisione umana;
- funzioni modulari di inference, valutazione e alert, riutilizzate sia sui dati reali sia in uno scenario dimostrativo;
- metriche adatte a un problema multiclasse;
- grafici interpretati con testo scritto;
- coda di revisione umana per i casi a bassa confidence;
- monitoraggio del sentiment con alert su baseline storica e su soglia di incremento, entrambi effettivamente utilizzati;
- bozza di pipeline CI/CD con un primo test di regressione, oltre allo smoke test, per repository GitHub.

I prossimi miglioramenti possibili restano: validare il modello su dati reali dell'azienda, raccogliere esempi corretti manualmente per un eventuale fine-tuning, sostituire l'euristica media+deviazione standard con un vero test statistico di drift (es. Kolmogorov-Smirnov, Population Stability Index), e collegare la pipeline CI/CD a un ambiente di deploy reale con versionamento di dati/modello e tracciamento degli esperimenti.